# REHAB24-6: 3D World Landmarks & Calibration Benchmark (`chamodi-week2`)

Validates **Variant B** (`angles_world.py`) and **Variant D** (`calibration.py`) against
REHAB24-6 OptiTrack 3D ground-truth data.

**Run cells 1 → 6 in order. No changes needed.**

In [1]:
# ============================================================
# STEP 1 — Python 3.12 environment for MediaPipe
# ============================================================
import os, subprocess

MP_PY = '/content/mpenv/bin/python'
if not os.path.exists(MP_PY):
    print('Installing uv and creating Python 3.12 env...')
    subprocess.run(['pip', 'install', '-q', 'uv'], check=True)
    subprocess.run(['uv', 'venv', '--python', '3.12', '/content/mpenv'], check=True)
    subprocess.run([
        'uv', 'pip', 'install', '--python', MP_PY,
        'numpy==1.26.4', 'mediapipe==0.10.14', 'pyyaml',
        'pandas', 'opencv-python', 'matplotlib',
        'scikit-learn', 'scipy',
    ], check=True)
print('MediaPipe Python 3.12 env ready at:', MP_PY)

Installing uv and creating Python 3.12 env...
MediaPipe Python 3.12 env ready at: /content/mpenv/bin/python


In [9]:
# ============================================================
# FULL RESTART CELL — run this after any kernel disconnect
# ============================================================
import glob, os, shutil, subprocess, sys, time, zipfile
import numpy as np, pandas as pd

# reset working directory FIRST — prevents "No such file or directory" error
os.chdir('/content')

# ---- core settings ----
REPO_DIR         = '/content/repo'
BRANCH           = 'main'
REPO_URL         = 'https://github.com/suvini2001/AI-based-Physiotherapy-project-.git'
MP_PY            = '/content/mpenv/bin/python'
DATA             = '/content/rehab24'
MAX_VIDEOS       = None
EXERCISES        = [1, 2, 3]
THRESHOLD        = 0.70
MARGIN           = 45
MAX_LAG          = 30
MODEL_COMPLEXITY = 1

ARMS_OVERRIDE = {
    'right': ('12: RightArm', '13: RightForeArm', '14: RightHand'),
    'left':  ('7: LeftArm',   '8: LeftForeArm',   '9: LeftHand'),
}

# ---- mount Drive ----
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
CACHE   = '/content/drive/MyDrive/rehab24_cache_3d'
RESULTS = '/content/drive/MyDrive/rehab24_results_3d'
for d in (DATA, CACHE, RESULTS):
    os.makedirs(d, exist_ok=True)

# ---- remove any old clone and clone correct branch ----
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
assert not os.path.exists(REPO_DIR), "Could not remove old repo"
subprocess.run(
    ['git', 'clone', '-q', '-b', BRANCH, REPO_URL, REPO_DIR],
    check=True, cwd='/content'
)
os.chdir(REPO_DIR)
for p in (REPO_DIR,
          os.path.join(REPO_DIR, 'capture'),
          os.path.join(REPO_DIR, 'pose'),
          os.path.join(REPO_DIR, 'vision')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('Branch:', BRANCH)

# ---- verify modules ----
import importlib.util
all_ok = True
for mod in ['angles', 'angles_world', 'benchmark_common',
            'calibration', 'filters', 'visibility']:
    spec = importlib.util.find_spec(mod)
    status = 'OK' if spec else 'MISSING!!!'
    if not spec: all_ok = False
    print(f'  {mod:20s}: {status}')
assert all_ok, "Some modules missing — check branch"

# ---- joint names ----
jn_files = glob.glob(f'{DATA}/**/*joint*names*.txt', recursive=True)
assert jn_files, "joints_names.txt not found — re-run Step 3 first"
with open(jn_files[0]) as f:
    JOINT_NAMES = [l.strip() for l in f if l.strip()]
print(f'Joint names: {len(JOINT_NAMES)} loaded')

for side, joints in ARMS_OVERRIDE.items():
    for jn in joints:
        assert jn in JOINT_NAMES, f"'{jn}' not in JOINT_NAMES"
print('ARMS_OVERRIDE: all joints verified OK')

# ---- segmentation ----
seg_files = glob.glob(f'{DATA}/**/Segmentation.*', recursive=True)
assert seg_files, "Segmentation.csv not found — re-run Step 3 first"
sep = ';' if ';' in open(seg_files[0]).readline() else ','
seg_df = pd.read_csv(seg_files[0], sep=sep)
cols_lower = {c.lower(): c for c in seg_df.columns}

def pick_col(candidates):
    for cand in candidates:
        for cl, orig in cols_lower.items():
            if cand in cl: return orig
    raise KeyError(f'No column matching {candidates}')

c_vid   = pick_col(['video_id', 'video', 'file', 'rec', 'name'])
c_ex    = pick_col(['exercise_id', 'exercise', 'activity', 'ex'])
c_start = pick_col(['first_frame', 'start', 'begin', 'f0', 'from'])
c_end   = pick_col(['last_frame',  'end',   'stop',  'f1', 'to'])
c_view  = None
for cl, orig in cols_lower.items():
    if any(k in cl for k in ['cam17_orientation', 'view', 'angle', 'cam_pos']):
        c_view = orig
        break
print(f'Columns: vid={c_vid}, ex={c_ex}, start={c_start}, end={c_end}, view={c_view}')

# ---- index files ----
all_videos = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.mp4', '.avi', '.mov'))
}
SKIP = {'joints_names.txt', 'joint_names.txt',
        'segmentation.csv', 'segmentation.txt'}
all_3d = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.npy', '.npz'))
    and os.path.basename(p).lower() not in SKIP
}
print(f'Videos: {len(all_videos)},  3D truth files: {len(all_3d)}')

# ---- build recordings ----
seg_ex = seg_df[seg_df[c_ex].isin(EXERCISES)].copy()
recordings = []
for (vid_id, ex_id), grp in seg_ex.groupby([c_vid, c_ex]):
    person_id = str(vid_id).strip()
    f0 = max(0, int(grp[c_start].min()) - MARGIN)
    f1 = int(grp[c_end].max()) + MARGIN
    view_val = str(grp[c_view].iloc[0]) if c_view else 'unknown'
    matched_vids = {k: v for k, v in all_videos.items()
                    if k.startswith(person_id + '-') or k == person_id}
    person_3d = {k: v for k, v in all_3d.items()
                 if k.startswith(person_id + '-') or k == person_id}
    if not person_3d: continue
    truth_key = next((k for k in person_3d if '120fps' in k), next(iter(person_3d)))
    for vid_key, vid_path in matched_vids.items():
        cam = ('cam18'  if 'cam18' in vid_key.lower() or 'transposed' in vid_key.lower()
               else 'cam17' if 'cam17' in vid_key.lower()
               else 'unknown')
        recordings.append({
            'rec_id':     f'{person_id}_ex{int(ex_id)}_{cam}',
            'video_key':  vid_key,
            'video_path': vid_path,
            'truth_path': person_3d[truth_key],
            'exercise':   int(ex_id),
            'f0': f0, 'f1': f1,
            'view_raw':   view_val,
            'cam':        cam,
        })

recordings = list({r['rec_id']: r for r in recordings}.values())
if MAX_VIDEOS:
    recordings = recordings[:MAX_VIDEOS]
print(f'Total recordings: {len(recordings)}')

# ---- cache status ----
cached_files = {os.path.basename(p) for p in glob.glob(f'{CACHE}/*.npz')}
matched = sum(1 for r in recordings
              if f"{r['rec_id']}_{r['f0']}_{r['f1']}_3d.npz" in cached_files)
print(f'Cache: {matched}/{len(recordings)} done — {len(recordings)-matched} remaining')
print('\nReady — now run Step 5')

Mounted at /content/drive


CalledProcessError: Command '['git', 'clone', '-q', '-b', 'main', 'https://github.com/suvini2001/AI-based-Physiotherapy-project-.git', '/content/repo']' returned non-zero exit status 128.

In [3]:
# ============================================================
# STEP 3 — Locate dataset in Google Drive
# ============================================================
import urllib.request

ZENODO_BASE = 'https://zenodo.org/records/13305826/files'

def find_in_drive(filename):
    if not os.path.exists('/content/drive/MyDrive'):
        return None
    matches = glob.glob(f'/content/drive/MyDrive/**/{filename}', recursive=True)
    return matches[0] if matches else None

def ensure_file(filename, url_name=None):
    target = os.path.join(DATA, filename)
    if os.path.exists(target) and os.path.getsize(target) > 0:
        print(f'[OK]       {filename}')
        return target
    hit = find_in_drive(filename)
    if hit:
        print(f'[Drive]    {filename} <- {hit}')
        if not os.path.exists(target):
            os.symlink(hit, target)
        return target
    url = f"{ZENODO_BASE}/{url_name or filename}?download=1"
    print(f'[Download] {url}')
    urllib.request.urlretrieve(url, target)
    return target

# --- Segmentation.csv ---
for name in ('Segmentation.csv', 'Segmentation.txt'):
    hit = find_in_drive(name)
    if hit:
        dst = os.path.join(DATA, os.path.basename(hit))
        if not os.path.exists(dst):
            shutil.copy(hit, dst)
        break
if not glob.glob(f'{DATA}/Segmentation.*'):
    try: ensure_file('Segmentation.csv')
    except: ensure_file('Segmentation.txt')

# --- joints_names.txt ---
for name in ('joints_names.txt', 'joint_names.txt'):
    hit = find_in_drive(name)
    if hit:
        dst = os.path.join(DATA, os.path.basename(hit))
        if not os.path.exists(dst):
            shutil.copy(hit, dst)
        break
if not glob.glob(f'{DATA}/joint*names*.txt'):
    ensure_file('joints_names.txt')

# --- videos (Ex1..Ex6 structure OR videos.zip) ---
# First check if videos already extracted into Ex* folders
existing_mp4s = glob.glob(f'{DATA}/**/*.mp4', recursive=True) + \
                glob.glob(f'{DATA}/**/*.avi', recursive=True)
if not existing_mp4s:
    # try Drive zip
    drive_zip = find_in_drive('videos.zip')
    if drive_zip:
        target = os.path.join(DATA, 'videos.zip')
        if not os.path.exists(target):
            os.symlink(drive_zip, target)
        print('Extracting videos.zip...')
        with zipfile.ZipFile(target) as zf:
            zf.extractall(DATA)
    else:
        z = ensure_file('videos.zip')
        print('Extracting videos.zip...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA)

# --- 3d_joints ---
existing_3d = glob.glob(f'{DATA}/3d_joints/**/*.*', recursive=True)
if not existing_3d:
    drive_3d_dir = None
    hits = glob.glob('/content/drive/MyDrive/**/3d_joints', recursive=True)
    if hits and os.path.isdir(hits[0]) and os.listdir(hits[0]):
        drive_3d_dir = hits[0]
    if drive_3d_dir:
        dst = os.path.join(DATA, '3d_joints')
        if not os.path.exists(dst):
            os.symlink(drive_3d_dir, dst)
        print(f'[Drive] Linked 3d_joints from {drive_3d_dir}')
    else:
        z = ensure_file('3d_joints.zip')
        print('Extracting 3d_joints.zip...')
        with zipfile.ZipFile(z) as zf:
            zf.extractall(DATA)

print('\nDATA contents:', os.listdir(DATA))
all_mp4s = glob.glob(f'{DATA}/**/*.mp4', recursive=True) + \
           glob.glob(f'{DATA}/**/*.avi', recursive=True)
print(f'Total video files found: {len(all_mp4s)}')
if all_mp4s:
    print('  Example:', all_mp4s[0])

Extracting videos.zip...
[Download] https://zenodo.org/records/13305826/files/3d_joints.zip?download=1


KeyboardInterrupt: 

In [4]:
import glob, numpy as np, os

cached = glob.glob(f'{CACHE}/*.npz')
good, bad = [], []
for p in cached:
    try:
        d = np.load(p)
        if 'lm' in d.files and d['lm'].shape[0] > 0:
            good.append(p)
        else:
            bad.append(p)
    except:
        bad.append(p)

print(f'Good cached files : {len(good)} / 70')
print(f'Broken cache files: {len(bad)}')
print(f'Remaining to extract: {70 - len(good)}')

Good cached files : 19 / 70
Broken cache files: 0
Remaining to extract: 51


In [5]:
import glob, os

cached = glob.glob(f'{CACHE}/*.npz')
cached_names = {os.path.basename(p) for p in cached}

print("=== Cached files (first 5) ===")
for p in sorted(cached)[:5]:
    print(" ", os.path.basename(p))

print("\n=== Expected cache names from recordings (first 5) ===")
for rec in recordings[:5]:
    expected = f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz"
    match = expected in cached_names
    print(f"  {expected}  ->  {'MATCH' if match else 'MISSING'}")

=== Cached files (first 5) ===
  PM_000_ex1_cam18_135_1310_3d.npz
  PM_000_ex1_unknown_135_1310_3d.npz
  PM_001_ex1_cam18_85_3006_3d.npz
  PM_001_ex1_unknown_85_3006_3d.npz
  PM_002_ex1_cam18_107_2012_3d.npz

=== Expected cache names from recordings (first 5) ===


NameError: name 'recordings' is not defined

In [ ]:
# ============================================================
# STEP 4 — Parse segmentation + build recordings list
# ============================================================
import numpy as np
import pandas as pd

# --- joint names ---
jn_files = glob.glob(f'{DATA}/**/*joint*names*.txt', recursive=True)
assert jn_files, 'joints_names.txt not found'
with open(jn_files[0]) as f:
    JOINT_NAMES = [l.strip() for l in f if l.strip()]
print(f'Loaded {len(JOINT_NAMES)} OptiTrack joint names')
print('  First 8:', JOINT_NAMES[:8])

# verify the arm joint names we need are present
for side, (sh, el, wr) in ARMS_OVERRIDE.items():
    for jn in (sh, el, wr):
        if jn not in JOINT_NAMES:
            print(f'  WARNING: {jn} not in JOINT_NAMES — check ARMS_OVERRIDE')
        else:
            print(f'  OK: {jn} found (idx {JOINT_NAMES.index(jn)})')

# --- segmentation ---
seg_files = glob.glob(f'{DATA}/**/Segmentation.*', recursive=True)
assert seg_files, 'Segmentation.csv not found'
sep = ';' if ';' in open(seg_files[0]).readline() else ','
seg_df = pd.read_csv(seg_files[0], sep=sep)
print('\nSegmentation columns:', list(seg_df.columns))
print(f'Total rows: {len(seg_df)}')

cols_lower = {c.lower(): c for c in seg_df.columns}

def pick_col(candidates):
    for cand in candidates:
        for cl, orig in cols_lower.items():
            if cand in cl:
                return orig
    raise KeyError(f'No column matching {candidates} in {list(seg_df.columns)}')

c_vid   = pick_col(['video_id', 'video', 'file', 'rec', 'name'])
c_ex    = pick_col(['exercise_id', 'exercise', 'activity', 'ex'])
c_start = pick_col(['first_frame', 'start', 'begin', 'f0', 'from'])
c_end   = pick_col(['last_frame', 'end', 'stop', 'f1', 'to'])

c_view = None
for cl, orig in cols_lower.items():
    if any(k in cl for k in ['cam17_orientation', 'view', 'angle', 'cam_pos']):
        c_view = orig
        break

print(f'Using columns: video={c_vid}, exercise={c_ex}, start={c_start}, end={c_end}, view={c_view}')

# --- index ALL video files (search whole DATA tree, not just videos/) ---
all_videos = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.mp4', '.avi', '.mov'))
}

# --- index 3D ground-truth files ---
all_3d = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob(f'{DATA}/**/*.*', recursive=True)
    if p.lower().endswith(('.npy', '.npz', '.txt', '.csv'))
    and os.path.basename(p).lower() not in
       ('joints_names.txt', 'joint_names.txt', 'segmentation.csv', 'segmentation.txt')
    and '3d_joints' in p
}

print(f'\nIndexed {len(all_videos)} video files')
print(f'Indexed {len(all_3d)} 3D ground-truth files')
if all_videos:
    print('  Video example:', list(all_videos.keys())[0])
if all_3d:
    print('  3D truth example:', list(all_3d.keys())[0])

# --- build recordings ---
seg_ex = seg_df[seg_df[c_ex].isin(EXERCISES)].copy()
recordings = []

for (vid_id, ex_id), grp in seg_ex.groupby([c_vid, c_ex]):
    vid_str  = str(vid_id).strip()
    base_id  = vid_str.split('-')[0]   # 'PM_013' from 'PM_013-cam17'
    f0       = max(0, int(grp[c_start].min()) - MARGIN)
    f1       = int(grp[c_end].max()) + MARGIN
    view_val = str(grp[c_view].iloc[0]) if c_view else 'unknown'

    # match video files by exact name or prefix
    matched_vids = [
        k for k in all_videos
        if k == vid_str
        or k.startswith(vid_str + '-')
        or k.startswith(base_id + '-')
    ]

    # match 3D truth by person ID (PM_013)
    truth_key = next((k for k in all_3d if base_id in k), None)

    if not truth_key:
        continue

    for mv in sorted(set(matched_vids)):
        recordings.append({
            'rec_id':     f'{mv}_ex{int(ex_id)}',
            'video_key':  mv,
            'video_path': all_videos[mv],
            'truth_path': all_3d[truth_key],
            'exercise':   int(ex_id),
            'f0': f0, 'f1': f1,
            'view_raw': view_val,
        })

# deduplicate
recordings = list({r['rec_id']: r for r in recordings}.values())
if MAX_VIDEOS is not None:
    recordings = recordings[:MAX_VIDEOS]

print(f'\nPrepared {len(recordings)} recordings for evaluation')
if not recordings:
    print('\n  !! 0 recordings found. Debug info:')
    print('  seg_df video_id sample:', seg_df[c_vid].unique()[:5])
    print('  all_videos keys sample:', list(all_videos.keys())[:5])
    print('  all_3d keys sample:    ', list(all_3d.keys())[:5])
else:
    for r in recordings[:3]:
        print(f"  {r['rec_id']}  view={r['view_raw']}")

In [ ]:
# ============================================================
# STEP 5 — Extract 3D landmarks + run benchmark
# ============================================================
import numpy as np

import angles as angles_2d
from angles_world import elbow_angle_3d, mask_world_like
import benchmark_common as bc
from calibration import fit_curve, apply_calibration
from filters import butterworth_filter
from gap_fill import fill_gaps as fill_gaps_max
from visibility import mask_low_visibility

print('All imports OK')

def infer_view(rec):
    v = str(rec.get('view_raw', '')).lower()
    if 'half' in v:                      return 'Half-profile'
    if 'front' in v or v == '0':         return 'Front'
    if 'prof' in v:                      return 'Profile'
    if 'side' in v or '90' in v:         return 'Side'
    if 'cam17' in rec['video_key'].lower(): return 'Front'
    if 'cam18' in rec['video_key'].lower(): return 'Side'
    return 'Unknown'

def get_3d_landmarks(rec):
    cache_path = os.path.join(
        CACHE, f"{rec['rec_id']}_{rec['f0']}_{rec['f1']}_3d.npz")
    need_extract = True
    if os.path.exists(cache_path):
        try:
            d = np.load(cache_path)
            if 'world' in d and 'lm' in d:
                need_extract = False
        except Exception:
            pass
    if need_extract:
        import os as _os
        _env = _os.environ.copy()
        _env['MPLBACKEND'] = 'Agg'          # prevent matplotlib_inline conflict
        _env['PYTHONPATH'] = REPO_DIR        # ensure pose/ is findable
        cmd = [MP_PY, 'extract_world.py',
               rec['video_path'],
               str(rec['f0']), str(rec['f1']),
               cache_path, str(MODEL_COMPLEXITY)]
        subprocess.run(cmd, check=True, env=_env, cwd=REPO_DIR)
    d = np.load(cache_path)
    return d['lm'], d['world'], tuple(d['size']), float(d['fps'])

rows_A, rows_B, aligned_cache = [], [], []

for idx, rec in enumerate(recordings, 1):
    try:
        t0 = time.time()
        lm, world, size, fps = get_3d_landmarks(rec)
        truth_arr  = bc.load_truth_array(rec['truth_path'], JOINT_NAMES)
        truth_win  = truth_arr[rec['f0']: rec['f0'] + len(world)]

        lm_masked    = mask_low_visibility(lm, threshold=THRESHOLD)
        world_masked = mask_world_like(world, lm_masked)

        for side in ('right', 'left'):
            arm_joints = ARMS_OVERRIDE[side]

            # ground truth 3D elbow angle
            gt_3d = bc.truth_elbow_3d(truth_win, JOINT_NAMES, arm_joints, use_dims=3)

            # Variant B — 3D world angle
            pred_b_raw  = elbow_angle_3d(world_masked, side=side)
            pred_b_fill = fill_gaps_max(pred_b_raw, max_gap=15)
            pred_b = (
                butterworth_filter(pred_b_fill, fs=fps, cutoff=6.0)
                if np.isfinite(pred_b_fill).sum() > 15 else pred_b_fill
            )

            # Variant A — 2D baseline
            pred_a_raw  = angles_2d.elbow_angle(lm_masked, side=side, size=size)
            pred_a_fill = fill_gaps_max(pred_a_raw, max_gap=15)
            pred_a = (
                butterworth_filter(pred_a_fill, fs=fps, cutoff=6.0)
                if np.isfinite(pred_a_fill).sum() > 15 else pred_a_fill
            )

            meta = {
                'recording': f"{rec['rec_id']}_{side}",
                'video':     rec['video_key'],
                'person':    bc.person_of(rec['video_key']),
                'exercise':  rec['exercise'],
                'side':      side,
                'view':      infer_view(rec),
                'max_lag':   MAX_LAG,
            }

            row_b = bc.make_row('B_3D_World',  pred_b, gt_3d, meta)
            row_a = bc.make_row('A_2D_Base',   pred_a, gt_3d, meta)
            if row_b: rows_B.append(row_b)
            if row_a: rows_A.append(row_a)

            # cache aligned predictions for LOPO calibration
            lag = bc.best_lag(pred_b, gt_3d, max_lag=MAX_LAG)
            if isinstance(lag, tuple):
                lag = lag[0]   # best_lag returns (lag, r)
            p_al, t_al = bc.overlap(pred_b, gt_3d, lag)
            aligned_cache.append({'meta': meta, 'pred_3d': p_al, 'truth_3d': t_al})

        mae_r = rows_B[-2]['mae'] if rows_B else float('nan')
        mae_l = rows_B[-1]['mae'] if len(rows_B) > 1 else float('nan')
        print(f'[{idx}/{len(recordings)}] {rec["rec_id"]}  {time.time()-t0:.1f}s'
              f'  3D MAE R/L: {mae_r:.1f}°/{mae_l:.1f}°')

    except Exception as e:
        print(f'[{idx}/{len(recordings)}] SKIPPED {rec["rec_id"]}: {e}')

# --- Variant D: Leave-One-Person-Out calibration on top of Variant B ---
rows_D = []
for item in aligned_cache:
    test_person = item['meta']['person']
    train_p = [x['pred_3d']  for x in aligned_cache if x['meta']['person'] != test_person]
    train_t = [x['truth_3d'] for x in aligned_cache if x['meta']['person'] != test_person]
    if train_p:
        p_cat = np.concatenate(train_p)
        t_cat = np.concatenate(train_t)
        valid = np.isfinite(p_cat) & np.isfinite(t_cat)
        if valid.sum() > 20:
            params = fit_curve(p_cat[valid], t_cat[valid])
            pred_cal = apply_calibration(item['pred_3d'], params)
        else:
            pred_cal = item['pred_3d']
    else:
        pred_cal = item['pred_3d']

    meta_d = dict(item['meta'], max_lag=0)
    row = bc.make_row('D_3D_Calibrated', pred_cal, item['truth_3d'], meta_d)
    if row: rows_D.append(row)

# --- save CSVs ---
bc.save_rows(rows_A, os.path.join(RESULTS, 'variant_A_2d.csv'))
bc.save_rows(rows_B, os.path.join(RESULTS, 'variant_B_3d_world.csv'))
bc.save_rows(rows_D, os.path.join(RESULTS, 'variant_D_3d_calibrated.csv'))
print(f'\nSaved CSVs to {RESULTS}')
print(f'Rows: A={len(rows_A)}, B={len(rows_B)}, D={len(rows_D)}')

In [ ]:
# ============================================================
# STEP 6 — Summary tables + chart
# ============================================================
import matplotlib.pyplot as plt
import benchmark_common as bc
import pandas as pd

df_all = bc.load_variants(os.path.join(RESULTS, 'variant_*.csv'))
print(f'Total rows loaded: {len(df_all)}')
print(f'Variants: {df_all["variant"].unique()}')

print('\n=== Summary (coverage >= 50%) ===')
sum_df = bc.summary(df_all, min_coverage=50.0)
print(sum_df.to_string(index=False))

print('\n=== Markdown table for notes.md ===')
md = bc.to_markdown(bc.results_table(df_all, min_coverage=50.0))
print(md)

sum_df.to_csv(os.path.join(RESULTS, 'summary_3d_by_view.csv'), index=False)

if not sum_df.empty and 'view' in sum_df.columns and 'mae' in sum_df.columns:
    fig, ax = plt.subplots(figsize=(9, 5))
    piv = sum_df.pivot(index='view', columns='variant', values='mae')
    piv.plot(kind='bar', ax=ax, rot=0)
    ax.set_ylabel('Median MAE (degrees)')
    ax.set_title('REHAB24-6 Elbow Angle Error: 2D vs 3D World vs 3D Calibrated')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    chart_path = os.path.join(RESULTS, 'mae_3d_comparison.png')
    plt.savefig(chart_path, dpi=150)
    plt.show()
    print('Saved chart to:', chart_path)

print('\n=== A vs B improvement (negative = B better) ===')
try:
    delta = bc.delta(df_all, 'B_3D_World', 'A_2D_Base')
    print(delta)
except Exception as e:
    print('delta not available:', e)